# Data pre-processing
This notebook contains all code which was used for data pre-processing and the Urban Heat Island Intensity calculation, inclunding:

- Access ESA CCI landcover data and retrieve landcover fractions
- Calculate daily temperature metrics
- Calculate ensemble member mean of five ensemble members of the IFS-FESOM storyline simulations
- Aggregate by city and cell type
- Add Köppen-Geiger climate classes and location
- Calculate city aggregated temperature metrics and Urban Heat Island Intensity
- Define percentile-based heat day threshold and consequently heatwave mask 

# Load packages

In [9]:
import pandas as pd
import xarray as xr
import numpy as np
import glob
import dask

import rioxarray as rxr
from rasterstats import zonal_stats
from shapely import geometry as geo
import tempfile

from pathlib import Path
import os

# Set working directories

In [2]:
save_script_to = "/home/b/b383801/"
save_data_to = "/work/bb1445/b383801/"
get_data_from = "/work/bb1152/m300755/inputdata/IFS-FESOM/"

# ESA Landcover data

In [2]:
#ds = xr.open_dataset("C3S-LC-L4-LCCS-Map-300m-P1Y-2019-v2.1.1.area-subset.80.80.30.-50.nc")

ds = xr.open_dataset("C3S-LC-L4-LCCS-Map-300m-P1Y-2019-v2.1.1.area-subset.80.80.30.-50.nc", chunks={"lat": 1000, "lon": 1000})
print(ds)

<xarray.Dataset> Size: 10GB
Dimensions:              (time: 1, lat: 18000, lon: 46800, bounds: 2)
Coordinates:
  * lat                  (lat) float64 144kB 80.0 80.0 79.99 ... 30.01 30.0 30.0
  * lon                  (lon) float64 374kB -50.0 -50.0 -49.99 ... 80.0 80.0
  * time                 (time) datetime64[ns] 8B 2019-01-01
Dimensions without coordinates: bounds
Data variables:
    lccs_class           (time, lat, lon) uint8 842MB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    processed_flag       (time, lat, lon) float32 3GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    current_pixel_state  (time, lat, lon) float32 3GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    observation_count    (time, lat, lon) uint16 2GB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    change_count         (time, lat, lon) uint8 842MB dask.array<chunksize=(1, 1000, 1000), meta=np.ndarray>
    crs                  int32 4B ...
    lat_bounds           (lat, b

C:\Users\lolel\AppData\Local\Temp\ipykernel_17532\716970935.py:3: UserWarning: The specified chunks separate the stored chunks along dimension "lat" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset("C3S-LC-L4-LCCS-Map-300m-P1Y-2019-v2.1.1.area-subset.80.80.30.-50.nc", chunks={"lat": 1000, "lon": 1000})
C:\Users\lolel\AppData\Local\Temp\ipykernel_17532\716970935.py:3: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset("C3S-LC-L4-LCCS-Map-300m-P1Y-2019-v2.1.1.area-subset.80.80.30.-50.nc", chunks={"lat": 1000, "lon": 1000})


## From ESA CCI Notebook
Click [here](https://climate.esa.int/de/projekte/land-cover/notebook-lc) to open. 

In [23]:
class_ids = ds["lccs_class"].attrs["flag_values"]
class_names = ds["lccs_class"].attrs["flag_meanings"].split()
class_colors = ds["lccs_class"].attrs["flag_colors"].split()

print(class_ids)
print(class_names)
print(class_colors)

[  0  10  11  12  20  30  40  50  60  61  62  70  71  72  80  81  82  90
 100 110 120 121 122 130 140 150 151 152 153 160 170 180 190 200 201 202
 210 220]
['no_data', 'cropland_rainfed', 'cropland_rainfed_herbaceous_cover', 'cropland_rainfed_tree_or_shrub_cover', 'cropland_irrigated', 'mosaic_cropland', 'mosaic_natural_vegetation', 'tree_broadleaved_evergreen_closed_to_open', 'tree_broadleaved_deciduous_closed_to_open', 'tree_broadleaved_deciduous_closed', 'tree_broadleaved_deciduous_open', 'tree_needleleaved_evergreen_closed_to_open', 'tree_needleleaved_evergreen_closed', 'tree_needleleaved_evergreen_open', 'tree_needleleaved_deciduous_closed_to_open', 'tree_needleleaved_deciduous_closed', 'tree_needleleaved_deciduous_open', 'tree_mixed', 'mosaic_tree_and_shrub', 'mosaic_herbaceous', 'shrubland', 'shrubland_evergreen', 'shrubland_deciduous', 'grassland', 'lichens_and_mosses', 'sparse_vegetation', 'sparse_tree', 'sparse_shrub', 'sparse_herbaceous', 'tree_cover_flooded_fresh_or_brakish

<div class="alert alert-block alert-info">
<b> Information:</b>
There is a missmatch between the length of class_ids / class_names and class_colors as there is one color code less than in the other variables. Therefor we need to add a color code for the class "no data" manually. 
</div>

In [24]:
if len(class_colors) < len(class_names):
    print("Missmatch in flag colors and meanings (too few colors)")
    class_colors = ["#000000"] + class_colors

Missmatch in flag colors and meanings (too few colors)


In [26]:
cmap = ListedColormap(class_colors)
norm = BoundaryNorm([v - 0.5 for v in class_ids] + [class_ids[-1] + 0.5], cmap.N)

## Add landcover data
Here, the CCI land cover data as well as the point based latitude and longitude will be added to the temperature data. 
The IFS-FESOM data is stored in an unstructured grid, which means that the distances between lat and lon values are not uniform (the grid is called HEALPix). When we open the original data, it is available as an array with 3.145.728 rows. Now we need the indices, labels, and land cover values of the cells we want to save from the 3.145.728 rows. Here is a table listing the lat and lon values:

`xyz = pd.read_pickle("/work/bb1152/m300755/inputdata/IFS-FESOM/case-studies/xzT.pkl")`

We will calculate the fraction of each land cover type in the surrounding of each data point. 

### Lat Lon Data

In [3]:
xyz = pd.read_pickle(get_data_from + "case-studies/xzT.pkl")
print(xyz)

              lat         lon           T
0        0.074604   45.000000  300.299446
1        0.149208   45.087891  300.376587
2        0.149208   44.912109  300.266785
3        0.223812   45.000000  300.326057
4        0.223812   45.175781  300.442734
...           ...         ...         ...
3145723 -0.223812  314.824219  298.619972
3145724 -0.223812  315.000000  299.300552
3145725 -0.149208  315.087891  299.008972
3145726 -0.149208  314.912109  298.934662
3145727 -0.074604  315.000000  298.666153

[3145728 rows x 3 columns]


### Landcover Data

In [4]:
lc = xr.open_dataset(get_data_from + "landcover/C3S-LC-L4-LCCS-Map-300m-P1Y-2019-v2.1.1.nc")
print(lc)

<xarray.Dataset> Size: 101GB
Dimensions:              (time: 1, lat: 64800, lon: 129600, bounds: 2)
Coordinates:
  * time                 (time) datetime64[ns] 8B 2019-01-01
  * lat                  (lat) float64 518kB 90.0 90.0 89.99 ... -90.0 -90.0
  * lon                  (lon) float64 1MB -180.0 -180.0 -180.0 ... 180.0 180.0
Dimensions without coordinates: bounds
Data variables:
    lccs_class           (time, lat, lon) uint8 8GB ...
    processed_flag       (time, lat, lon) float32 34GB ...
    current_pixel_state  (time, lat, lon) float32 34GB ...
    observation_count    (time, lat, lon) uint16 17GB ...
    change_count         (time, lat, lon) uint8 8GB ...
    crs                  int32 4B ...
    lat_bounds           (lat, bounds) float64 1MB ...
    lon_bounds           (lon, bounds) float64 2MB ...
    time_bounds          (time, bounds) datetime64[ns] 16B ...
Attributes: (12/38)
    title:                      Land Cover Map of 2019
    summary:                    This dat

sh: getfattr: command not found


### Add Buffer circle arround each point
We will use a fixed buffer size of <b> 4500 m radius </b> in the 3857 crs. 

In [ ]:
# make buffer circles around each point
def buffer_circles(xyz_gdf, radius_m=4500, crs_m=3857):
    original_crs = xyz_gdf.crs
    xyz_gdf_m = xyz_gdf.to_crs(epsg=crs_m)
    buffer_m = xyz_gdf_m.geometry.buffer(radius_m)
    xyz_gdf_out = xyz_gdf_m.to_crs(original_crs)
    buffer_gdf = gpd.GeoDataFrame(geometry=buffer_m, crs=f"EPSG:{crs_m}")
    xyz_gdf_out["buffer"] = buffer_gdf.to_crs(original_crs).geometry
    
    return xyz_gdf_out

xyz_gdf = buffer_circles(xyz_gdf, radius_m=8500)
print(xyz_gdf.head())

### Calculate land cover fraction for each buffer circle / point 

In [3]:
def calculate_lc_fractions_rasterstats(xyz_gdf,
                                       lc_data,                      
                                       lc_variable="lccs_class",
                                       time_index=0,
                                       batch_size=10000,
                                       save_path=None, 
                                       output_name=None): 

    original_crs = xyz_gdf.crs
    
    if "time" in lc_data.dims:
        lc = lc_data.isel(time=time_index)
    else:
        lc = lc_data
        
    if hasattr(lc, lc_variable):
        lc_array = lc[lc_variable]
    else:
        lc_array = lc

    class_code_to_name = {}
    if hasattr(lc_array, "flag_values") and hasattr(lc_array, "flag_meanings"):
        lc_class_codes = lc_array.flag_values
        lc_class_meanings = lc_array.flag_meanings
        
        if isinstance(lc_class_meanings, str):
            meanings_list = lc_class_meanings.split()
        else:
            meanings_list = lc_class_meanings
        class_code_to_name = dict(zip(lc_class_codes, meanings_list))
    else:
        print("Warning: No flag_values or flag_meanings found. Using numeric codes.")

    if not hasattr(lc_array, "rio"):
        raise ValueError("lc_data must have rioxaaray accessor.")

    if lc_array.rio.crs is None:
        lc_array = lc_array.rio.write_crs(original_crs)

    # rasterstats need GeoTIFF
    tmpf = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
    tmp_tif = tmpf.name
    tmpf.close()
    lc_array.rio.to_raster(tmp_tif)
    print("Created temporary file:", tmp_tif)

    lc_raster = rxr.open_rasterio(tmp_tif, masked=True)
    if "band" in lc_raster.dims:
        lc_raster = lc_raster.isel(band=0)

    raster_crs = lc_raster.rio.crs
    raster_nodata = lc_raster.rio.nodata
    affine = lc_raster.rio.transform()

    if xyz_gdf.crs is None:
        raise ValueError("xyz_gdf has no CRS set.")

    if "buffer" not in xyz_gdf.columns:
        raise ValueError("xyz_gdf must contain a buffer column with polygons.")

    if xyz_gdf.crs != raster_crs:
        xyz_gdf_proj = xyz_gdf.to_crs(raster_crs)
    else:
        xyz_gdf_proj = xyz_gdf.copy()

    n_total = len(xyz_gdf_proj)
    n_batches = int(np.ceil(n_total / batch_size))
    print(f"Processing {n_total} buffers in {n_batches} batches")

    results = []
    all_class_codes = set()

    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, n_total)
        batch_gdf = xyz_gdf_proj.iloc[start_idx:end_idx]

        buffer_geoms = list(batch_gdf["buffer"].values)

        zonal_results = zonal_stats(
            buffer_geoms,
            tmp_tif,
            affine=affine, 
            nodata=raster_nodata,
            categorical=True,
            all_touched=False)     # only pixels included that have their centre in polygon

        for i, stats_dict in enumerate(zonal_results):
            orig_idx = batch_gdf.index[i]
            row = batch_gdf.iloc[i]

            stats_clean = {int(k): int(v) for k, v in stats_dict.items() if k is not None and (raster_nodata is None or int(k) != raster_nodata)}

            total_pixels = sum(stats_clean.values())
            all_class_codes.update(stats_clean.keys())

            if total_pixels > 0:
                fractions = {int(k): v / total_pixels for k, v in stats_clean.items()}
            else:
                fractions = {}

            fraction_cols = {}
            for class_code, fraction_value in fractions.items():
                if class_code in class_code_to_name:
                    col_name = f"{class_code_to_name[class_code]}_fraction"
                else:
                    col_name = f"lc_class_{class_code}_fraction"
                fraction_cols[col_name] = fraction_value

            result_dict = {
                "point_index": orig_idx,
                "lat": row["lat"] if "lat" in row.index else row["buffer"].centroid.y,
                "lon": row["lon"] if "lon" in row.index else row["buffer"].centroid.x,
                "total_pixels": total_pixels,
                **fraction_cols}
            results.append(result_dict)

        if (batch_idx + 1) % 10 == 0 or batch_idx == n_batches - 1:
            print(f"Processed batch {batch_idx + 1}/{n_batches} (buffers {start_idx}-{end_idx})")

    try:
        os.remove(tmp_tif)
        print("Removed temporary raster:", tmp_tif)
    except Exception as e:
        print("Could not remove temporary file:", tmp_tif)

    result_df = pd.DataFrame(results).fillna(0)

    fraction_cols = [col for col in result_df.columns if col.endswith("_fraction")]
    if len(fraction_cols) > 0:
        max_fraction_col = result_df[fraction_cols].idxmax(axis=1)
        result_df["main_lc_class"] = max_fraction_col.apply(lambda x: x.replace("_fraction", ""))
    else:
        result_df["main_lc_class"] = None

    full_path = os.path.join(save_path, output_name)
    result_df.to_csv(full_path, index=False)
    
    return result_df

In [ ]:
lc_fractions_globally = calculate_lc_fractions_rasterstats(xyz_gdf, lc, lc_variable="lccs_class", time_index=0,
                                                          save_path=save_data_to, output_name="lc/lc_fractions_globally.csv")

### Load saved data

In [7]:
lc_fractions_globally = pd.read_csv(save_data_to + "lc/lc_fractions_globally.csv")
lc_fractions_globally

,point_index,lat,lon,total_pixels,water_fraction,cropland_rainfed_fraction,grassland_fraction,bare_areas_fraction,bare_areas_consolidated_fraction,bare_areas_unconsolidated_fraction,...,sparse_tree_fraction,tree_mixed_fraction,tree_needleleaved_evergreen_closed_fraction,snow_and_ice_fraction,shrubland_evergreen_fraction,tree_needleleaved_deciduous_closed_fraction,tree_needleleaved_evergreen_open_fraction,lichens_and_mosses_fraction,tree_needleleaved_deciduous_open_fraction,main_lc_class
0,0,0.074604,45.000000,2364,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
1,1,0.149208,45.087891,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
2,2,0.149208,44.912109,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3,3,0.223812,45.000000,2374,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
4,4,0.223812,45.175781,2372,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145723,3145723,-0.223812,-45.175781,2372,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3145724,3145724,-0.223812,-45.000000,2374,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3145725,3145725,-0.149208,-44.912109,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3145726,3145726,-0.149208,-45.087891,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water


In [8]:
# Printing counts of each main land cover class
print(lc_fractions_globally["main_lc_class"].value_counts())

main_lc_class
water                                         2258949
bare_areas                                     118170
snow_and_ice                                    89812
tree_broadleaved_evergreen_closed_to_open       87462
grassland                                       86139
shrubland                                       69716
cropland_rainfed                                54852
sparse_vegetation                               52989
cropland_rainfed_herbaceous_cover               48457
tree_needleleaved_evergreen_closed_to_open      43423
tree_broadleaved_deciduous_closed_to_open       37793
tree_needleleaved_deciduous_closed_to_open      36568
tree_broadleaved_deciduous_open                 25464
tree_needleleaved_evergreen_closed              19998
cropland_irrigated                              15107
mosaic_cropland                                 13408
mosaic_tree_and_shrub                           13255
shrubland_deciduous                             12884
tree_mixed    

# Ensemble Model Mean

## Loading multiple files at once with xarray using dask

In [4]:
def load_ifs_fesom_data(data_dir, 
                        pattern="*.nc", 
                        chunks={"time": 12}): 
    
    file_pattern = str(Path(data_dir) / pattern)
    files = sorted(glob.glob(file_pattern))
    
    print(f"Found {len(files)} files")
    print(f"First file: {files[0]}")
    print(f"Last file: {files[-1]}")
    
    ds = xr.open_mfdataset(files,
                           combine="by_coords",  
                           parallel=True,         
                           chunks=chunks,         
                           engine="netcdf4")
    
    print(f"\nCombined dataset:")
    print(f"Time range: {ds.time.min().values} to {ds.time.max().values}")
    print(f"Shape: {ds.dims}")
    print(f"Size in memory (if loaded): {ds.nbytes / 1e9:.2f} GB")
    
    return ds

## Calculate daily statistics (mean, max, min of temperature)

In [5]:
def calculate_daily_stats_dask_simple(ds,  
                                      n_workers=4, 
                                      save_path=None, 
                                      output_name=None):
    
    dask.config.set(scheduler="threads", num_workers=n_workers)
    
    print(f"Using {n_workers} parallel workers...")
    print("Calculating daily statistics...")
    
    daily_mean = ds.resample(time='1D').mean(dim='time').compute()
    daily_max = ds.resample(time='1D').max(dim='time').compute()
    daily_min = ds.resample(time='1D').min(dim='time').compute()
    
    daily_stats = xr.Dataset({"T2M_mean": daily_mean,
                              "T2M_max": daily_max,
                              "T2M_min": daily_min,
                              "lat": ds["lat"],
                              "lon": ds["lon"]})
    
    daily_stats_computed = daily_stats.compute()
    
    full_path = os.path.join(save_path, output_name)
    daily_stats_computed.to_netcdf(full_path)
    return daily_stats_computed

## Get data

### IFS-FESOM HIST (all ensemble-member)

In [8]:
hist_urb_r1 = load_ifs_fesom_data(get_data_from + "hist/urban/r1", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
hist_urb_r2 = load_ifs_fesom_data(get_data_from + "hist/urban/r2", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
hist_urb_r3 = load_ifs_fesom_data(get_data_from + "hist/urban/r3", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
hist_urb_r4 = load_ifs_fesom_data(get_data_from + "hist/urban/r4", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
hist_urb_r5 = load_ifs_fesom_data(get_data_from + "hist/urban/r5", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})

Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/hist/urban/r1/IFS-FESOM_story-nudging_high_2t_2017_01.nc
Last file: /work/bb1152/m300755/inputdata/IFS-FESOM/hist/urban/r1/IFS-FESOM_story-nudging_high_2t_2026_04.nc

Combined dataset:
  Time range: 2017-01-01T00:00:00.000000000 to 2026-04-30T23:00:00.000000000
  Shape: FrozenMappingWarningOnValuesAccess({'time': 81768, 'point_index': 8909})
  Size in memory (if loaded): 5.83 GB
Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/hist/urban/r2/IFS-FESOM_story-nudging_high_2t_2017_01.nc
Last file: /work/bb1152/m300755/inputdata/IFS-FESOM/hist/urban/r2/IFS-FESOM_story-nudging_high_2t_2026_04.nc

Combined dataset:
  Time range: 2017-01-01T00:00:00.000000000 to 2026-04-30T23:00:00.000000000
  Shape: FrozenMappingWarningOnValuesAccess({'time': 81768, 'point_index': 8909})
  Size in memory (if loaded): 5.83 GB
Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/hist/urban/r3/IFS-FESOM_story

## IFS-FESOM CONT (all ensemble-member)

In [9]:
cont_urb_r1 = load_ifs_fesom_data(get_data_from + "cont/urban/r1", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
cont_urb_r2 = load_ifs_fesom_data(get_data_from + "cont/urban/r2", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
cont_urb_r3 = load_ifs_fesom_data(get_data_from + "cont/urban/r3", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
cont_urb_r4 = load_ifs_fesom_data(get_data_from + "cont/urban/r4", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
cont_urb_r5 = load_ifs_fesom_data(get_data_from + "cont/urban/r5", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})

Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/cont/urban/r1/IFS-FESOM_story-nudging_high_2t_2017_01.nc
Last file: /work/bb1152/m300755/inputdata/IFS-FESOM/cont/urban/r1/IFS-FESOM_story-nudging_high_2t_2026_04.nc

Combined dataset:
  Time range: 2017-01-01T00:00:00.000000000 to 2026-04-30T23:00:00.000000000
  Shape: FrozenMappingWarningOnValuesAccess({'time': 81768, 'point_index': 8909})
  Size in memory (if loaded): 5.83 GB
Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/cont/urban/r2/IFS-FESOM_story-nudging_high_2t_2017_01.nc
Last file: /work/bb1152/m300755/inputdata/IFS-FESOM/cont/urban/r2/IFS-FESOM_story-nudging_high_2t_2026_04.nc

Combined dataset:
  Time range: 2017-01-01T00:00:00.000000000 to 2026-04-30T23:00:00.000000000
  Shape: FrozenMappingWarningOnValuesAccess({'time': 81768, 'point_index': 8909})
  Size in memory (if loaded): 5.83 GB
Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/cont/urban/r3/IFS-FESOM_story

## IFS-FESOM T2K (all ensemble-member)

In [10]:
T2K_urb_r1 = load_ifs_fesom_data(get_data_from + "Tplus2.0K/urban/r1", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
T2K_urb_r2 = load_ifs_fesom_data(get_data_from + "Tplus2.0K/urban/r2", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
T2K_urb_r3 = load_ifs_fesom_data(get_data_from + "Tplus2.0K/urban/r3", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
T2K_urb_r4 = load_ifs_fesom_data(get_data_from + "Tplus2.0K/urban/r4", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})
T2K_urb_r5 = load_ifs_fesom_data(get_data_from + "Tplus2.0K/urban/r5", pattern="IFS-FESOM_story-nudging_high_2t_*.nc", chunks={'time': 12})

Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/Tplus2.0K/urban/r1/IFS-FESOM_story-nudging_high_2t_2017_01.nc
Last file: /work/bb1152/m300755/inputdata/IFS-FESOM/Tplus2.0K/urban/r1/IFS-FESOM_story-nudging_high_2t_2026_04.nc

Combined dataset:
  Time range: 2017-01-01T00:00:00.000000000 to 2026-04-30T23:00:00.000000000
  Shape: FrozenMappingWarningOnValuesAccess({'time': 81768, 'point_index': 8909})
  Size in memory (if loaded): 5.83 GB
Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/Tplus2.0K/urban/r2/IFS-FESOM_story-nudging_high_2t_2017_01.nc
Last file: /work/bb1152/m300755/inputdata/IFS-FESOM/Tplus2.0K/urban/r2/IFS-FESOM_story-nudging_high_2t_2026_04.nc

Combined dataset:
  Time range: 2017-01-01T00:00:00.000000000 to 2026-04-30T23:00:00.000000000
  Shape: FrozenMappingWarningOnValuesAccess({'time': 81768, 'point_index': 8909})
  Size in memory (if loaded): 5.83 GB
Found 112 files
First file: /work/bb1152/m300755/inputdata/IFS-FESOM/Tplus2.0K

## Access temperature data without loading into memory

In [12]:
temperature_hist_r1 = hist_urb_r1["T2M"]
temperature_hist_r2 = hist_urb_r2["T2M"]
temperature_hist_r3 = hist_urb_r3["T2M"]
temperature_hist_r4 = hist_urb_r4["T2M"]
temperature_hist_r5 = hist_urb_r5["T2M"]

In [14]:
temperature_cont_r1 = cont_urb_r1["T2M"]
temperature_cont_r2 = cont_urb_r2["T2M"]
temperature_cont_r3 = cont_urb_r3["T2M"]
temperature_cont_r4 = cont_urb_r4["T2M"]
temperature_cont_r5 = cont_urb_r5["T2M"]

In [15]:
temperature_2K_r1 = T2K_urb_r1["T2M"]
temperature_2K_r2 = T2K_urb_r2["T2M"]
temperature_2K_r3 = T2K_urb_r3["T2M"]
temperature_2K_r4 = T2K_urb_r4["T2M"]
temperature_2K_r5 = T2K_urb_r5["T2M"]

## Calculate daily mean, max and min temperature
<div class="alert alert-box alert-success"> Using our <b>`calculate_daily_stats_dask_simple()` function</b>! </div> Only for all urban pixels, not applying any smoothing.

### HIST

In [16]:
daily_stats_hist_urb_r1 = calculate_daily_stats_dask_simple(temperature_hist_r1, save_path=save_data_to, output_name="urban/r1/daily_stats_hist_r1.nc")
daily_stats_hist_urb_r2 = calculate_daily_stats_dask_simple(temperature_hist_r2, save_path=save_data_to, output_name="urban/r2/daily_stats_hist_r2.nc")
daily_stats_hist_urb_r3 = calculate_daily_stats_dask_simple(temperature_hist_r3, save_path=save_data_to, output_name="urban/r3/daily_stats_hist_r3.nc")
daily_stats_hist_urb_r4 = calculate_daily_stats_dask_simple(temperature_hist_r4, save_path=save_data_to, output_name="urban/r4/daily_stats_hist_r4.nc")
daily_stats_hist_urb_r5 = calculate_daily_stats_dask_simple(temperature_hist_r5, save_path=save_data_to, output_name="urban/r5/daily_stats_hist_r5.nc")

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

### CONT

In [17]:
daily_stats_cont_urb_r1 = calculate_daily_stats_dask_simple(temperature_cont_r1, save_path=save_data_to, output_name="urban/r1/daily_stats_cont_r1.nc")
daily_stats_cont_urb_r2 = calculate_daily_stats_dask_simple(temperature_cont_r2, save_path=save_data_to, output_name="urban/r2/daily_stats_cont_r2.nc")
daily_stats_cont_urb_r3 = calculate_daily_stats_dask_simple(temperature_cont_r3, save_path=save_data_to, output_name="urban/r3/daily_stats_cont_r3.nc")
daily_stats_cont_urb_r4 = calculate_daily_stats_dask_simple(temperature_cont_r4, save_path=save_data_to, output_name="urban/r4/daily_stats_cont_r4.nc")
daily_stats_cont_urb_r5 = calculate_daily_stats_dask_simple(temperature_cont_r5, save_path=save_data_to, output_name="urban/r5/daily_stats_cont_r5.nc")

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

### T2K

In [ ]:
daily_stats_T2K_urb_r1 = calculate_daily_stats_dask_simple(temperature_2K_r1, save_path=save_data_to, output_name="urban/r1/daily_stats_T2K_r1.nc")
daily_stats_T2K_urb_r2 = calculate_daily_stats_dask_simple(temperature_2K_r2, save_path=save_data_to, output_name="urban/r2/daily_stats_T2K_r2.nc")
daily_stats_T2K_urb_r3 = calculate_daily_stats_dask_simple(temperature_2K_r3, save_path=save_data_to, output_name="urban/r3/daily_stats_T2K_r3.nc")
daily_stats_T2K_urb_r4 = calculate_daily_stats_dask_simple(temperature_2K_r4, save_path=save_data_to, output_name="urban/r4/daily_stats_T2K_r4.nc")
daily_stats_T2K_urb_r5 = calculate_daily_stats_dask_simple(temperature_2K_r5, save_path=save_data_to, output_name="urban/r5/daily_stats_T2K_r5.nc")

Using 4 parallel workers...
Calculating daily statistics...


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

Using 4 parallel workers...
Calculating daily statistics...


### Load saved data

#### HIST

In [24]:
daily_stats_hist_urb_r1 = xr.open_dataset(save_data_to + "urban/r1/daily_stats_hist_r1.nc")
daily_stats_hist_urb_r2 = xr.open_dataset(save_data_to + "urban/r2/daily_stats_hist_r2.nc")
daily_stats_hist_urb_r3 = xr.open_dataset(save_data_to + "urban/r3/daily_stats_hist_r3.nc")
daily_stats_hist_urb_r4 = xr.open_dataset(save_data_to + "urban/r4/daily_stats_hist_r4.nc")
daily_stats_hist_urb_r5 = xr.open_dataset(save_data_to + "urban/r5/daily_stats_hist_r5.nc")

In [22]:
daily_stats_hist_urb_r1

<xarray.Dataset> Size: 729MB
Dimensions:      (time: 3407, point_index: 8909)
Coordinates:
  * time         (time) datetime64[ns] 27kB 2017-01-01 2017-01-02 ... 2026-04-30
  * point_index  (point_index) int64 71kB 393655 393658 393671 ... 884715 884716
    lat          (point_index) float64 71kB ...
    lon          (point_index) float64 71kB ...
Data variables:
    T2M_mean     (time, point_index) float64 243MB ...
    T2M_max      (time, point_index) float64 243MB ...
    T2M_min      (time, point_index) float64 243MB ...

#### CONT

In [25]:
daily_stats_cont_urb_r1 = xr.open_dataset(save_data_to + "urban/r1/daily_stats_cont_r1.nc")
daily_stats_cont_urb_r2 = xr.open_dataset(save_data_to + "urban/r2/daily_stats_cont_r2.nc")
daily_stats_cont_urb_r3 = xr.open_dataset(save_data_to + "urban/r3/daily_stats_cont_r3.nc")
daily_stats_cont_urb_r4 = xr.open_dataset(save_data_to + "urban/r4/daily_stats_cont_r4.nc")
daily_stats_cont_urb_r5 = xr.open_dataset(save_data_to + "urban/r5/daily_stats_cont_r5.nc")

#### T2K

In [27]:
daily_stats_T2K_urb_r1 = xr.open_dataset(save_data_to + "urban/r1/daily_stats_T2K_r1.nc")
daily_stats_T2K_urb_r2 = xr.open_dataset(save_data_to + "urban/r2/daily_stats_T2K_r2.nc")
daily_stats_T2K_urb_r3 = xr.open_dataset(save_data_to + "urban/r3/daily_stats_T2K_r3.nc")
daily_stats_T2K_urb_r4 = xr.open_dataset(save_data_to + "urban/r4/daily_stats_T2K_r4.nc")
daily_stats_T2K_urb_r5 = xr.open_dataset(save_data_to + "urban/r5/daily_stats_T2K_r5.nc")

## Calculate model member mean 

In [33]:
def calculate_ensemble_member_mean(ds_r1, ds_r2, ds_r3, ds_r4, ds_r5,
                                   save_path=None,
                                   output_name=None):
    combined = xr.concat([ds_r1, ds_r2, ds_r3, ds_r4, ds_r5], dim="member")
    ensemble_mean = combined.mean(dim="member")

    if save_path is not None:
        full_path = os.path.join(save_path, output_name)
        ensemble_mean.to_netcdf(full_path)
        
    return ensemble_mean

### HIST

In [35]:
daily_stats_hist_urb_ens_mean = calculate_ensemble_member_mean(daily_stats_hist_urb_r1,
                                                               daily_stats_hist_urb_r2,
                                                               daily_stats_hist_urb_r3,
                                                               daily_stats_hist_urb_r4,
                                                               daily_stats_hist_urb_r5)

In [44]:
daily_stats_hist_urb_ens_mean.to_netcdf(save_data_to + "urban/ensemble_mean/daily_stats_hist_ens_mean.nc")

HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

### CONT

In [45]:
daily_stats_cont_urb_ens_mean = calculate_ensemble_member_mean(daily_stats_cont_urb_r1,
                                                               daily_stats_cont_urb_r2,
                                                               daily_stats_cont_urb_r3,
                                                               daily_stats_cont_urb_r4,
                                                               daily_stats_cont_urb_r5,
                                                              save_path=save_data_to,
                                                              output_name="urban/ensemble_mean/daily_stats_cont_ens_mean.nc")

HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

### T2K

In [46]:
daily_stats_T2K_urb_ens_mean = calculate_ensemble_member_mean(daily_stats_T2K_urb_r1,
                                                               daily_stats_T2K_urb_r2,
                                                               daily_stats_T2K_urb_r3,
                                                               daily_stats_T2K_urb_r4,
                                                               daily_stats_T2K_urb_r5,
                                                              save_path=save_data_to,
                                                              output_name="urban/ensemble_mean/daily_stats_T2K_ens_mean.nc")

HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/work/bb1445/b38380

### Load saved datasets

In [47]:
daily_stats_hist_urb_ens_mean = xr.open_dataset(save_data_to + "urban/ensemble_mean/daily_stats_hist_ens_mean.nc")
daily_stats_cont_urb_ens_mean = xr.open_dataset(save_data_to + "urban/ensemble_mean/daily_stats_cont_ens_mean.nc")
daily_stats_T2K_urb_ens_mean = xr.open_dataset(save_data_to + "urban/ensemble_mean/daily_stats_T2K_ens_mean.nc")

# Add city names, Köppen-Geiger climate class, location depending on hemisphere

## Functions

### Add city names and cell types to temperature data

In [4]:
def add_city_classification(daily_stats,
                            city_id,
                            save_path=None,
                            output_name=None, 
                            unknown_label: str = "unknown",
                            verbose: bool = True,):

    records = []
    for city in city_id.city.values:
        for cell_type in city_id.cell_type.values:
            grid_ids = city_id["grid_ids"].sel(
                city=city, cell_type=cell_type).values

            if grid_ids is None:
                continue

            grid_ids = np.asarray(grid_ids)

            if grid_ids.ndim == 0:
                grid_ids = grid_ids.item()

            grid_ids = np.atleast_1d(grid_ids)

            for gid in grid_ids:
                records.append({
                    "point_index": int(gid),
                    "city": city,
                    "cell_type": cell_type})

    mapping_df = pd.DataFrame(records).drop_duplicates("point_index")

    points_df = pd.DataFrame({
        "point_index": daily_stats.point_index.values})

    merged_df = points_df.merge(
        mapping_df, on="point_index", how="left")

    merged_df["city"] = merged_df["city"].fillna(unknown_label)
    merged_df["cell_type"] = merged_df["cell_type"].fillna(unknown_label)

    daily_stats_out = daily_stats.assign_coords(
        city=("point_index", merged_df["city"].to_numpy(dtype="U")),
        cell_type=("point_index", merged_df["cell_type"].to_numpy(dtype="U")),)

    daily_stats_out["city"].attrs = {"long_name": "City name",
                                     "description": "City to which this grid point belongs",}

    daily_stats_out["cell_type"].attrs = {"long_name": "Urbanization class",
                                          "description": "Urban classification based on land-cover fractions",
                                          "categories": "urban_high, urban_low, rural",}

    if verbose:
        total = len(merged_df)
        matched = (merged_df["city"] != unknown_label).sum()

        print("\nCity classification summary")
        print("-" * 35)
        print(f"Total points        : {total:,}")
        print(f"Classified points   : {matched:,}")
        print(f"Unclassified points : {total - matched:,}")
        print(f"Coverage            : {matched / total * 100:.2f}%")

        print("\nCell type distribution:")
        print(merged_df["cell_type"].value_counts())

        print("\nTop 10 cities:")
        print(merged_df["city"].value_counts().head(10))

    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        daily_stats_out.to_netcdf(full_path, mode="w")

    return daily_stats_out

### Add Köppen-Geiger climate class (main & detailed) to data

In [5]:
def  add_koppen_to_temperature_data(temp_ds, 
                                    koppen_ds, 
                                    save_path=None, 
                                    output_name=None):

    climate_groups = {
        1: 'A', 2: 'A', 3: 'A', 4: 'A',  # Tropical
        5: 'B', 6: 'B', 7: 'B', 8: 'B',  # Arid
        9: 'C', 10: 'C', 11: 'C', 12: 'C', 13: 'C', 14: 'C', 15: 'C', 16: 'C', 17: 'C',  # Temperate
        18: 'D', 19: 'D', 20: 'D', 21: 'D', 22: 'D', 23: 'D', 24: 'D', 25: 'D', 26: 'D', 27: 'D', 28: 'D', 29: 'D',  # Continental
        30: 'E', 31: 'E'  # Polar
    }
    
    koppen_names = {
        1: 'Af', 2: 'Am', 3: 'Aw', 4: 'As',
        5: 'BWh', 6: 'BWk', 7: 'BSh', 8: 'BSk',
        9: 'Csa', 10: 'Csb', 11: 'Csc', 12: 'Cwa', 13: 'Cwb', 14: 'Cwc',
        15: 'Cfa', 16: 'Cfb', 17: 'Cfc',
        18: 'Dsa', 19: 'Dsb', 20: 'Dsc', 21: 'Dsd',
        22: 'Dwa', 23: 'Dwb', 24: 'Dwc', 25: 'Dwd',
        26: 'Dfa', 27: 'Dfb', 28: 'Dfc', 29: 'Dfd',
        30: 'ET', 31: 'EF'
    }
    
    print("Adding Köppen-Geiger climate classifications to temperature data...")
    print(f"Processing {len(temp_ds.point_index)} points...")
    
    # Get Köppen data
    kg_class = koppen_ds['kg_class'].values
    kg_confidence = koppen_ds['kg_confidence'].values if 'kg_confidence' in koppen_ds else None
    lons = koppen_ds['lon'].values
    lats = koppen_ds['lat'].values
    
    # Get temperature data coordinates
    temp_lats = temp_ds.lat.values
    temp_lons = temp_ds.lon.values
    
    # Initialize arrays for climate classifications
    n_points = len(temp_ds.point_index)
    kg_class_main = np.empty(n_points, dtype='U10') 
    kg_class_detailed = np.empty(n_points, dtype='U10')  
    kg_conf = np.empty(n_points, dtype=np.float32) if kg_confidence is not None else None
    
    # Match each temperature point to Köppen class
    for i, (lat, lon) in enumerate(zip(temp_lats, temp_lons)):
        lat_idx = np.argmin(np.abs(lats - lat))
        lon_idx = np.argmin(np.abs(lons - lon))
        
        kg_value = kg_class[lat_idx, lon_idx]
        
        if np.isnan(kg_value) or kg_value == 0:
            kg_class_main[i] = 'Unknown'
            kg_class_detailed[i] = 'Unknown'
            if kg_conf is not None:
                kg_conf[i] = 0.0
        else:
            kg_int = int(kg_value)
            kg_class_main[i] = climate_groups.get(kg_int, 'Unknown')
            kg_class_detailed[i] = koppen_names.get(kg_int, 'Unknown')
            
            if kg_conf is not None:
                kg_conf[i] = kg_confidence[lat_idx, lon_idx]
        
        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1}/{n_points} points...")
    
    # Add as coordinates to the dataset
    temp_ds = temp_ds.assign_coords({
        'kg_class_main': ('point_index', kg_class_main),
        'kg_class_detailed': ('point_index', kg_class_detailed)
    })
    
    if kg_conf is not None:
        temp_ds = temp_ds.assign_coords({
            'kg_confidence': ('point_index', kg_conf)
        })
    
    # Add attributes
    temp_ds['kg_class_main'].attrs = {
        'long_name': 'Köppen-Geiger main climate group',
        'description': 'Main climate classification: A=Tropical, B=Arid, C=Temperate, D=Continental, E=Polar',
        'source': 'Köppen-Geiger climate classification'
    }
    
    temp_ds['kg_class_detailed'].attrs = {
        'long_name': 'Köppen-Geiger detailed climate class',
        'description': 'Detailed Köppen-Geiger climate classification',
        'source': 'Köppen-Geiger climate classification'
    }
    
    if kg_conf is not None:
        temp_ds['kg_confidence'].attrs = {
            'long_name': 'Köppen-Geiger classification confidence',
            'description': 'Confidence level of the Köppen-Geiger classification (%)',
            'units': '%'
        }
    
    # Print summary statistics
    print("\n" + "="*60)
    print("Climate Classification Summary:")
    print("="*60)
    
    print("\nMain climate groups:")
    unique_main, counts_main = np.unique(kg_class_main, return_counts=True)
    for group, count in zip(unique_main, counts_main):
        pct = (count / n_points) * 100
        print(f"  {group}: {count:,} points ({pct:.1f}%)")
    
    print("\nDetailed climate classes:")
    unique_detailed, counts_detailed = np.unique(kg_class_detailed, return_counts=True)
    sorted_idx = np.argsort(counts_detailed)[::-1]
    for idx in sorted_idx[:15]:
        clim_class = unique_detailed[idx]
        count = counts_detailed[idx]
        pct = (count / n_points) * 100
        print(f"  {clim_class}: {count:,} points ({pct:.1f}%)")
    
    if len(unique_detailed) > 15:
        print(f"  ... and {len(unique_detailed) - 15} more classes")
    
    if kg_conf is not None:
        mean_conf = np.nanmean(kg_conf)
        print(f"\nMean classification confidence: {mean_conf:.1f}%")

    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        temp_ds.to_netcdf(full_path)
    
    return temp_ds

### Add location depending on hemisphere to data

In [6]:
def add_location(ds,
                save_path=None,
                output_name=None):
    location_list = []
    for lat in ds.lat.values:
        if lat < 0:
            location_list.append("SH")
        else:
            location_list.append("NH")

    # Add as coordinate
    ds = ds.assign_coords({
        'location': (['point_index'], np.array(location_list, dtype="U2"))
    })

    # Add attributes
    ds['location'].attrs={
        'long_name': 'Location depending on latitude',
        'description': 'Location: NH (northern hemisphere), SH (southern hemisphere)'      
    }

    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        ds.to_netcdf(full_path)

    return ds

## Get data

### City indices

In [7]:
city_id = pd.read_pickle(get_data_from + "landcover/cell_ids.pkl")
city_id = city_id.to_xarray()
print(city_id)

<xarray.Dataset> Size: 7kB
Dimensions:    (city: 225, cell_type: 3)
Coordinates:
  * city       (city) object 2kB 'Addis Ababa' ... 'Fernando de la Mora'
  * cell_type  (cell_type) object 24B 'urban_high' 'urban_low' 'rural'
Data variables:
    grid_ids   (city, cell_type) object 5kB [9194] ... [3115312 3115301 31153...


### Land cover

In [8]:
lc_fractions_globally = pd.read_csv(save_data_to + "lc/lc_fractions_globally.csv")
lc_fractions_globally

,point_index,lat,lon,total_pixels,water_fraction,cropland_rainfed_fraction,grassland_fraction,bare_areas_fraction,bare_areas_consolidated_fraction,bare_areas_unconsolidated_fraction,...,sparse_tree_fraction,tree_mixed_fraction,tree_needleleaved_evergreen_closed_fraction,snow_and_ice_fraction,shrubland_evergreen_fraction,tree_needleleaved_deciduous_closed_fraction,tree_needleleaved_evergreen_open_fraction,lichens_and_mosses_fraction,tree_needleleaved_deciduous_open_fraction,main_lc_class
0,0,0.074604,45.000000,2364,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
1,1,0.149208,45.087891,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
2,2,0.149208,44.912109,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3,3,0.223812,45.000000,2374,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
4,4,0.223812,45.175781,2372,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145723,3145723,-0.223812,-45.175781,2372,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3145724,3145724,-0.223812,-45.000000,2374,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3145725,3145725,-0.149208,-44.912109,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water
3145726,3145726,-0.149208,-45.087891,2371,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,water


### Köppen-Geiger climate class data

In [9]:
cc_kogei = xr.open_dataset(save_data_to + "cc_Köppen_Geiger/1991_2020/koppen_geiger_0p00833333.nc")

### IFS-FESOM output 
<b>Aggregated daily</b> output for urban cells: Ensemble member mean.
For calculating the ensemble mean of each variable the hourly input data was first agregrated to daily data for each model ensemble member separately. Afterwards the ensemble mean of each variable was calculated for each day. See ```01a_Model_Member_Mean.ipynb```.

In [10]:
daily_stats_hist_urb = xr.open_dataset(save_data_to + "urban/ensemble_mean/daily_stats_hist_ens_mean.nc")
daily_stats_cont_urb = xr.open_dataset(save_data_to + "urban/ensemble_mean/daily_stats_cont_ens_mean.nc")
daily_stats_t2k_urb = xr.open_dataset(save_data_to + "urban/ensemble_mean/daily_stats_T2K_ens_mean.nc")

In [11]:
daily_stats_hist_urb

<xarray.Dataset> Size: 729MB
Dimensions:      (time: 3407, point_index: 8909)
Coordinates:
  * time         (time) datetime64[ns] 27kB 2017-01-01 2017-01-02 ... 2026-04-30
  * point_index  (point_index) int64 71kB 393655 393658 393671 ... 884715 884716
    lat          (point_index) float64 71kB ...
    lon          (point_index) float64 71kB ...
Data variables:
    T2M_mean     (time, point_index) float64 243MB ...
    T2M_max      (time, point_index) float64 243MB ...
    T2M_min      (time, point_index) float64 243MB ...

## Add cell type 
Based on `city_id` data set (`city_ids.pkl`)

- Combining `daily_stats_*_urb` datasets with `city_id`-> for each point define <b> city class </b> (urban high / low density or rural).
- Assign <b> city names </b> to each point based on `city` coordinate of `city_id` data. 

### Checking how much of the point indices (`grid_ids`) of the `city_id` data are included in the temperature data

In [12]:
ids_daily = set(daily_stats_hist_urb.point_index.values)

ids_city = set()
for arr in city_id["grid_ids"].values.ravel():
    if arr is None:
        continue
    for gid in arr:
        ids_city.add(int(gid))

print("daily_stats points:", len(ids_daily))
print("city_class grid_ids:", len(ids_city))

overlap = ids_daily & ids_city
print("overlap:", len(overlap))
print("overlap %:", len(overlap) / len(ids_daily) * 100)

print("example overlaps:", list(overlap)[:10])
print("example only in daily:", list(ids_daily - ids_city)[:10])
print("example only in city:", list(ids_city - ids_daily)[:10])

daily_stats points: 8909
city_class grid_ids: 8886
overlap: 8886
overlap %: 99.74183410034796
example overlaps: [393655, 393658, 393671, 1540558, 393679, 393682, 393686, 393687, 393688, 393689]
example only in daily: [np.int64(178948), np.int64(178949), np.int64(873996), np.int64(178990), np.int64(178607), np.int64(178991), np.int64(189494), np.int64(178618), np.int64(178619), np.int64(178622)]
example only in city: []


## Add city names and cell type classification of `city_id` to temperature data 
<div class="alert alert-box alert-success"> Using our <b>`add_city_classification()` function</b>! </div>

### HIST

In [13]:
daily_stats_hist_urb_enhanced = add_city_classification(daily_stats_hist_urb, city_id)


City classification summary
-----------------------------------
Total points        : 8,909
Classified points   : 8,886
Unclassified points : 23
Coverage            : 99.74%

Cell type distribution:
cell_type
rural         5661
urban_low     2726
urban_high     499
unknown         23
Name: count, dtype: int64

Top 10 cities:
city
Pudong         110
Tokyo          107
Chicago         96
Los Angeles     94
Tianjin         93
New York        89
Zhengzhou       88
London          83
Beijing         83
Jakarta         80
Name: count, dtype: int64


### CONT

In [15]:
daily_stats_cont_urb_enhanced = add_city_classification(daily_stats_cont_urb, city_id)


City classification summary
-----------------------------------
Total points        : 8,909
Classified points   : 8,886
Unclassified points : 23
Coverage            : 99.74%

Cell type distribution:
cell_type
rural         5661
urban_low     2726
urban_high     499
unknown         23
Name: count, dtype: int64

Top 10 cities:
city
Pudong         110
Tokyo          107
Chicago         96
Los Angeles     94
Tianjin         93
New York        89
Zhengzhou       88
London          83
Beijing         83
Jakarta         80
Name: count, dtype: int64


### T2K

In [16]:
daily_stats_t2k_urb_enhanced = add_city_classification(daily_stats_t2k_urb, city_id)


City classification summary
-----------------------------------
Total points        : 8,909
Classified points   : 8,886
Unclassified points : 23
Coverage            : 99.74%

Cell type distribution:
cell_type
rural         5661
urban_low     2726
urban_high     499
unknown         23
Name: count, dtype: int64

Top 10 cities:
city
Pudong         110
Tokyo          107
Chicago         96
Los Angeles     94
Tianjin         93
New York        89
Zhengzhou       88
London          83
Beijing         83
Jakarta         80
Name: count, dtype: int64


## Add Köppen-Geiger climate class 
<div class="alert alert-block alert-success"> Using our `<b>add_koppen_to_temperature_data</b>` function!</div>

### HIST

In [17]:
daily_stats_hist_urb_enhanced = add_koppen_to_temperature_data(temp_ds = daily_stats_hist_urb_enhanced, koppen_ds=cc_kogei)

Adding Köppen-Geiger climate classifications to temperature data...
Processing 8909 points...
  Processed 1000/8909 points...
  Processed 2000/8909 points...
  Processed 3000/8909 points...
  Processed 4000/8909 points...
  Processed 5000/8909 points...
  Processed 6000/8909 points...
  Processed 7000/8909 points...
  Processed 8000/8909 points...

Climate Classification Summary:

Main climate groups:
  A: 1,627 points (18.3%)
  B: 1,329 points (14.9%)
  C: 3,727 points (41.8%)
  D: 2,224 points (25.0%)
  Unknown: 2 points (0.0%)

Detailed climate classes:
  Cwc: 2,011 points (22.6%)
  Cfa: 899 points (10.1%)
  Dfa: 761 points (8.5%)
  Dwd: 740 points (8.3%)
  Aw: 653 points (7.3%)
  BSh: 586 points (6.6%)
  Dsd: 582 points (6.5%)
  As: 463 points (5.2%)
  Csc: 388 points (4.4%)
  BSk: 369 points (4.1%)
  BWk: 318 points (3.6%)
  Am: 277 points (3.1%)
  Af: 234 points (2.6%)
  Csa: 221 points (2.5%)
  Cwa: 175 points (2.0%)
  ... and 9 more classes

Mean classification confidence: 91.2

### CONT

In [18]:
daily_stats_cont_urb_enhanced = add_koppen_to_temperature_data(temp_ds = daily_stats_cont_urb_enhanced, koppen_ds=cc_kogei)

Adding Köppen-Geiger climate classifications to temperature data...
Processing 8909 points...
  Processed 1000/8909 points...
  Processed 2000/8909 points...
  Processed 3000/8909 points...
  Processed 4000/8909 points...
  Processed 5000/8909 points...
  Processed 6000/8909 points...
  Processed 7000/8909 points...
  Processed 8000/8909 points...

Climate Classification Summary:

Main climate groups:
  A: 1,627 points (18.3%)
  B: 1,329 points (14.9%)
  C: 3,727 points (41.8%)
  D: 2,224 points (25.0%)
  Unknown: 2 points (0.0%)

Detailed climate classes:
  Cwc: 2,011 points (22.6%)
  Cfa: 899 points (10.1%)
  Dfa: 761 points (8.5%)
  Dwd: 740 points (8.3%)
  Aw: 653 points (7.3%)
  BSh: 586 points (6.6%)
  Dsd: 582 points (6.5%)
  As: 463 points (5.2%)
  Csc: 388 points (4.4%)
  BSk: 369 points (4.1%)
  BWk: 318 points (3.6%)
  Am: 277 points (3.1%)
  Af: 234 points (2.6%)
  Csa: 221 points (2.5%)
  Cwa: 175 points (2.0%)
  ... and 9 more classes

Mean classification confidence: 91.2

### T2K

In [19]:
daily_stats_t2k_urb_enhanced = add_koppen_to_temperature_data(temp_ds = daily_stats_t2k_urb_enhanced, koppen_ds=cc_kogei)

Adding Köppen-Geiger climate classifications to temperature data...
Processing 8909 points...
  Processed 1000/8909 points...
  Processed 2000/8909 points...
  Processed 3000/8909 points...
  Processed 4000/8909 points...
  Processed 5000/8909 points...
  Processed 6000/8909 points...
  Processed 7000/8909 points...
  Processed 8000/8909 points...

Climate Classification Summary:

Main climate groups:
  A: 1,627 points (18.3%)
  B: 1,329 points (14.9%)
  C: 3,727 points (41.8%)
  D: 2,224 points (25.0%)
  Unknown: 2 points (0.0%)

Detailed climate classes:
  Cwc: 2,011 points (22.6%)
  Cfa: 899 points (10.1%)
  Dfa: 761 points (8.5%)
  Dwd: 740 points (8.3%)
  Aw: 653 points (7.3%)
  BSh: 586 points (6.6%)
  Dsd: 582 points (6.5%)
  As: 463 points (5.2%)
  Csc: 388 points (4.4%)
  BSk: 369 points (4.1%)
  BWk: 318 points (3.6%)
  Am: 277 points (3.1%)
  Af: 234 points (2.6%)
  Csa: 221 points (2.5%)
  Cwa: 175 points (2.0%)
  ... and 9 more classes

Mean classification confidence: 91.2

## Add location based on hemisphere
<div class="alert alert-block alert-success"> Using our `<b>add_location</b>` function!</div>

### HIST

In [20]:
for coord in daily_stats_hist_urb_enhanced.coords:
    print(coord, daily_stats_hist_urb_enhanced[coord].dtype)

point_index int64
lat float64
lon float64
time datetime64[ns]
city <U19
cell_type <U10
kg_class_main <U10
kg_class_detailed <U10
kg_confidence float32


In [21]:
for var in daily_stats_hist_urb_enhanced.variables:
    print(var, daily_stats_hist_urb_enhanced[var].dtype)

T2M_mean float64
T2M_max float64
T2M_min float64
point_index int64
lat float64
lon float64
time datetime64[ns]
city <U19
cell_type <U10
kg_class_main <U10
kg_class_detailed <U10
kg_confidence float32


In [25]:
daily_stats_hist_urb_enhanced = add_location(daily_stats_hist_urb_enhanced,
                                            save_path=save_data_to, output_name="urban/ensemble_mean/include_city_class/daily_stats_hist_urb_enhanced.nc")

In [26]:
daily_stats_hist_urb_enhanced

<xarray.Dataset> Size: 731MB
Dimensions:            (time: 3407, point_index: 8909)
Coordinates:
  * time               (time) datetime64[ns] 27kB 2017-01-01 ... 2026-04-30
  * point_index        (point_index) int64 71kB 393655 393658 ... 884715 884716
    lat                (point_index) float64 71kB 22.43 22.35 ... 53.67 53.67
    lon                (point_index) float64 71kB 113.4 112.9 ... -1.496 -1.266
    city               (point_index) <U19 677kB 'Foshan' ... 'Sheffield'
    cell_type          (point_index) <U10 356kB 'urban_low' ... 'urban_low'
    kg_class_main      (point_index) <U10 356kB 'C' 'C' 'C' 'A' ... 'C' 'C' 'C'
    kg_class_detailed  (point_index) <U10 356kB 'Csc' 'Csc' ... 'Cfa' 'Cfa'
    kg_confidence      (point_index) float32 36kB 75.0 50.0 ... 100.0 100.0
    location           (point_index) <U2 71kB 'NH' 'NH' 'NH' ... 'NH' 'NH' 'NH'
Data variables:
    T2M_mean           (time, point_index) float64 243MB ...
    T2M_max            (time, point_index) float64 243MB ...
    T2M_min            (time, point_index) float64 243MB ...

### CONT

In [27]:
daily_stats_cont_urb_enhanced = add_location(daily_stats_cont_urb_enhanced,
                                            save_path=save_data_to, output_name="urban/ensemble_mean/include_city_class/daily_stats_cont_urb_enhanced.nc")

### T2K

In [28]:
daily_stats_t2k_urb_enhanced = add_location(daily_stats_t2k_urb_enhanced,
                                           save_path=save_data_to, output_name="urban/ensemble_mean/include_city_class/daily_stats_t2k_urb_enhanced.nc")

# City aggregated temperature metrics (including Urban Heat Island Intensity)

## Functions

### Calculate UHII, keeping T_min, T_max, T_mean of each cell_type and city

Taking `T2M_mean_mean_urban_high` and `T2M_mean_mean_rural` for UHII calculation.

In [4]:
def calculate_uhii_extended(ds,
                            to_celsius=False,
                            exclude_leap_years=True,
                            save_path=None,
                            output_name=None):

    ds = ds.where(ds.city != "unknown", drop=True)

    cities = np.unique(ds.city.values)
    print(f"Processing {len(cities)} cities")
    print(f"Time period: {ds.time.min().values} to {ds.time.max().values}")

    def aggregate_temp(var, cell_type, agg):
        return (
            ds[var]
            .where(ds.cell_type == cell_type)
            .groupby(ds.city)
            .reduce(agg, dim="point_index")
        )

    stats = {}

    for var in ["T2M_mean", "T2M_max", "T2M_min"]:
        for cell in ["urban_high", "urban_low", "rural"]:
            stats[f"{var}_mean_{cell}"] = aggregate_temp(var, cell, np.nanmean)
            stats[f"{var}_max_{cell}"]  = aggregate_temp(var, cell, np.nanmax)
            stats[f"{var}_min_{cell}"]  = aggregate_temp(var, cell, np.nanmin)


    uhii = (stats["T2M_mean_mean_urban_high"] - stats["T2M_mean_mean_rural"])
    uhii.name = "UHII"

    n_urban_high = ((ds.cell_type == "urban_high").groupby(ds.city).sum(dim="point_index"))
    n_rural = ((ds.cell_type == "rural").groupby(ds.city).sum(dim="point_index"))

    location_by_city = ds['location'].groupby(ds.city).first()
    kg_class_main_by_city = ds['kg_class_main'].groupby(ds.city).first()
    kg_class_detailed_by_city = ds['kg_class_detailed'].groupby(ds.city).first()

    kg_confidence_by_city = ds['kg_confidence'].groupby(ds.city).mean(dim='point_index')
    lat_by_city = ds['lat'].groupby(ds.city).mean(dim='point_index')
    lon_by_city = ds['lon'].groupby(ds.city).mean(dim='point_index')

    uhii_ds = xr.Dataset(stats)
    uhii_ds["UHII"] = uhii
    uhii_ds["n_urban_high_points"] = n_urban_high
    uhii_ds["n_rural_points"] = n_rural
    uhii_ds["location"] = location_by_city
    uhii_ds["kg_class_main"] = kg_class_main_by_city
    uhii_ds["kg_class_detailed"] = kg_class_detailed_by_city
    uhii_ds["kg_confidence_city"] = kg_confidence_by_city
    uhii_ds["lat"] = lat_by_city
    uhii_ds["lon"] = lon_by_city

    coords_to_set = ['kg_class_main', 'kg_class_detailed', 
                     'kg_confidence_city', 'lat', 'lon', 'location']
    
    for coord in coords_to_set:
        if coord in uhii_ds:
            uhii_ds = uhii_ds.set_coords(coord)

    if to_celsius:
        for v in uhii_ds.data_vars:
            if v.startswith("T2M"):
                uhii_ds[v] = uhii_ds[v] - 273.15
                uhii_ds["UHII"].attrs["units"] = "°C"
    else:
        uhii_ds["UHII"].attrs["units"] = "K"

    uhii_ds["UHII"].attrs.update({
        "long_name": "Urban Heat Island Intensity",
        "description": "Mean urban_high temperature minus mean rural temperature",
        "formula": "mean(T_urban_high) - mean(T_rural)"
    })

    uhii_ds["n_urban_high_points"].attrs["long_name"] = "Number of urban_high grid points per city"
    uhii_ds["n_rural_points"].attrs["long_name"] = "Number of rural grid points per city"
    uhii_ds["location"].attrs["long_name"] = "Hemisphere location"
    uhii_ds["location"].attrs["description"] = "NH for Northern Hemisphere, SH for Southern Hemisphere"

    uhii_ds.coords["kg_class_main"].attrs["long_name"] = "Köppen climate classification (main class)"
    uhii_ds.coords["kg_class_main"].attrs["description"] = "A=Tropical, B=Arid, C=Temperate, D=Continental, E=Polar"
    
    uhii_ds.coords["kg_class_detailed"].attrs["long_name"] = "Köppen climate classification (detailed)"
    
    uhii_ds.coords["kg_confidence_city"].attrs["long_name"] = "Mean Köppen classification confidence for city"
    uhii_ds.coords["kg_confidence_city"].attrs["description"] = "Mean confidence across all points in city"
    
    uhii_ds.coords["lat"].attrs["long_name"] = "Mean latitude of city"
    uhii_ds.coords["lat"].attrs["units"] = "degrees_north"
    
    uhii_ds.coords["lon"].attrs["long_name"] = "Mean longitude of city"
    uhii_ds.coords["lon"].attrs["units"] = "degrees_east"

    if exclude_leap_years:
        print("Excluding leap days (February 29)")
        is_not_leap_day = ~((uhii_ds.time.dt.month == 2) & (uhii_ds.time.dt.day == 29))
        uhii_ds = uhii_ds.isel(time=is_not_leap_day)
        n_removed = (~is_not_leap_day).sum().values
        print(f"Removed {n_removed} leap days")
    
    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        uhii_ds.to_netcdf(full_path)

    return uhii_ds

### Calculate heat day thresholds & heat day mask
Calulated for `T2M_mean_mean_urban_high`and `T2M_mean_mean_rural` & Using <b>thresholds calculated based on HIST</b> for CONT and T2K

 ADJUSTED VERSION (POOLING INSTEAD OF SMOOTHING)
 takes into account all 15 (or more) days around each caler day for each year (not mean of those 15 days)

In [7]:
def identify_extreme_heat_pooling(ds, 
                                  variables=["T2M_mean_mean_urban_high", "T2M_mean_mean_rural"],
                                  percentiles=[90,95,98],
                                  window_days=15,
                                  exclude_leap_years=True,
                                  save_path=None,
                                  output_name=None,
                                  use_int8=True,
                                  reference_thresholds=None):
    
    print(f"Calculating extreme heat days for {variables}")

    if exclude_leap_years:
        print("Excluding leap days (February 29)")
        is_not_leap_day = ~((ds.time.dt.month == 2) & (ds.time.dt.day == 29))
        ds = ds.isel(time=is_not_leap_day)
        n_removed = (~is_not_leap_day).sum().values
        print(f"Removed {n_removed} leap days")

    month = ds.time.dt.month.values
    day = ds.time.dt.day.values
    ref_dates = pd.to_datetime([f"2017-{m:02d}-{d:02d}" for m, d in zip(month, day)])
    doy_clean = xr.DataArray(ref_dates.dayofyear, coords={"time": ds.time}, dims="time")

    extreme_days = xr.Dataset(coords=ds.coords)
    percentile_clim = {}

    for var_name in variables:
        print(f"\nProcessing variable: {var_name}")
        var = ds[var_name]
        var = var.assign_coords(dayofyear=doy_clean)

        percentile_clim[var_name] = {}
    
        if reference_thresholds is None:
            for p in percentiles: 
                print(f"Calculating {p}th percentile")
                with warnings.catch_warnings():
                    warnings.filterwarnings(
                        "ignore",
                        message="All-NaN slice encountered",
                        category=RuntimeWarning,)
                    half = window_days // 2
                    clim_list = []
                    for doy in range (1, 366):
                        window_doys = [(doy - half + i - 1) % 365 + 1
                                      for i in range(window_days)]
                        mask = var.dayofyear.isin(window_doys)
                        pooled = var.isel(time=mask)
                        q = pooled.quantile(p / 100.0, dim="time").drop_vars("quantile")
                        q = q.assign_coords(dayofyear=doy)
                        clim_list.append(q)
                    clim = xr.concat(clim_list, dim="dayofyear")
               
                percentile_clim[var_name][p] = clim    
        else: 
            percentile_clim[var_name] = reference_thresholds[var_name]
    
        for p in percentiles:
            print(f"Identifying days exceeding {p}th percentile")
            threshold = percentile_clim[var_name][p].sel(dayofyear=var.dayofyear)
    
            is_extreme = var >= threshold 
    
            if use_int8:
                is_extreme_save = is_extreme.astype(np.int8)
                dtype_note = "int8 (0=False, 1=True)"
            else:
                is_extreme_save = is_extreme
                dtype_note = "bool"
                
            extreme_days[f"{var_name}_extreme_p{p}"] = is_extreme_save.drop_vars("dayofyear")
            extreme_days[f"{var_name}_extreme_p{p}"].attrs = {"long_name": f"Extremeheat day ({var_name} > {p}th percentile)",
                                                                "description": f"{var_name} exceeds {p}th percentile for this day of year",
                                                                "data_type": dtype_note,
                                                                "leap_days_excluded": int(exclude_leap_years),
                                                                "threshold_source": "current_dataset" if reference_thresholds is None else "reference_dataset"}
            extreme_days[f"{var_name}_threshold_p{p}"] = threshold.drop_vars("dayofyear")
            extreme_days[f"{var_name}_threshold_p{p}"].attrs = {"long_name": f"{p}th percentile threshold"} # , "units": variables.attrs.get("units", "K")
    
    print("\nSummary of extreme days:")
    for var_name in variables:
        for p in percentiles:
            da = extreme_days[f"{var_name}_extreme_p{p}"]
            n_extreme = da.sum().values
            total = da.size
            pct = (n_extreme / total) * 100 
            print(f"{var_name}, {p}th percentile: {n_extreme:,} days ({pct:.2f}%)")

    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        extreme_days.to_netcdf(full_path)

    return extreme_days, percentile_clim

###  Calculate heatwave mask 
From `T2M_mean_mean_urban_high` and `T2M_mean_mean_rural` with labeling of all consecutive heat days as heatwave

In [8]:
def identify_heatwaves(ds_heat,
                       variables=["T2M_mean_mean_urban_high_extreme_p90", "T2M_mean_mean_urban_high_extreme_p95", "T2M_mean_mean_urban_high_extreme_p98",
                                 "T2M_mean_mean_rural_extreme_p90", "T2M_mean_mean_rural_extreme_p95", "T2M_mean_mean_rural_extreme_p98"],
                       min_consecutive_days=3,
                       use_int8=True,
                       save_path=None,
                       output_name=None):

    def _identify_heatwave_sequence(heat_days, min_days):
        n = len(heat_days)
        heatwave_days = np.zeros(n, dtype=bool)
        i = 0
        while i < n:
            if heat_days[i]:
                start = i
                while i < n and heat_days[i]:
                    i += 1
                end = i
                sequence_length = end - start
                if sequence_length >= min_days:
                    heatwave_days[start:end] = True
            else:
                i += 1
        return heatwave_days
        
    if isinstance(variables, str):
        variables = [variables]

    hw_ds = xr.Dataset(coords=ds_heat.coords)

    for var in variables:
        da = ds_heat[var]

        da_bool = da.fillna(0).astype(bool)

        hw = xr.full_like(da_bool, False, dtype=bool)
        other_dims = [dim for dim in da.dims if dim != 'time']

        if len(other_dims) == 0: 
            hw.values[:] = _identify_heatwave_sequence(da_bool.values, min_consecutive_days)
        else: 
            stacked = da_bool.stack(location=other_dims)
            hw_stacked = hw.stack(location=other_dims)

            for i in range(stacked.sizes['location']):
                heat_days = stacked.isel(location=i).values
                hw_stacked.values[:, i] = _identify_heatwave_sequence(heat_days, min_consecutive_days)

            hw = hw_stacked.unstack('location')

        hw = hw.fillna(False)

        hw_out = hw.astype(np.int8) if use_int8 else hw

        newname = var + "_heatwave"
        hw_out.name = newname

        hw_out.attrs = {"long_name": f"Heatwave indicator from {var}",
                        "description": f"{min_consecutive_days} consecutive extreme heat days",
                        "data_type": "int8 (0=no, 1=yes)" if use_int8 else "bool"}

        hw_ds[newname] = hw_out

    print("\nSummary of heatwaves:")
    for var in variables:
        da = ds_heat[var]
        da_bool = da.fillna(0).astype(bool)
        hw = hw_ds[var+ "_heatwave"].astype(bool)
        
        n_heat_days = da_bool.sum().values
        n_heatwave_days = hw.sum().values
        total_days = da_bool.size
        pct_hw_of_total = (n_heatwave_days / total_days) * 100 
        pct_hw_of_heatdays = ((n_heatwave_days / n_heat_days) * 100
                                if n_heat_days > 0 else 0.0)
        
        print(f"Extreme heat days total ({var}): {n_heat_days:,}")
        print(f"Heatwave days (>= {min_consecutive_days} consecutive) ({var}): {n_heatwave_days:,}")
        print(f"Percentage of all days that are heatwave days ({var}): {pct_hw_of_total:.2f}%")
        print(f"Percentage of heat days that belong to heatwaves ({var}): {pct_hw_of_heatdays:.2f}%")

    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        hw_ds.to_netcdf(full_path, mode="w")

    return hw_ds

## Get data

### Daily statistics (mean, max, min T) with all coords

In [4]:
daily_stats_hist_urb = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/daily_stats_hist_urb_enhanced.nc")
daily_stats_cont_urb = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/daily_stats_cont_urb_enhanced.nc")
daily_stats_t2k_urb = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/daily_stats_t2k_urb_enhanced.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


In [13]:
print(daily_stats_hist_urb)

<xarray.Dataset> Size: 730MB
Dimensions:            (time: 3407, point_index: 8909)
Coordinates:
  * time               (time) datetime64[ns] 27kB 2017-01-01 ... 2026-04-30
  * point_index        (point_index) int64 71kB 393655 393658 ... 884715 884716
    lat                (point_index) float64 71kB ...
    lon                (point_index) float64 71kB ...
    city               (point_index) <U19 677kB ...
    cell_type          (point_index) <U10 356kB ...
    kg_class_main      (point_index) <U7 249kB ...
    kg_class_detailed  (point_index) <U7 249kB ...
    kg_confidence      (point_index) float32 36kB ...
    location           (point_index) <U2 71kB ...
Data variables:
    T2M_mean           (time, point_index) float64 243MB ...
    T2M_max            (time, point_index) float64 243MB ...
    T2M_min            (time, point_index) float64 243MB ...


## Calculate city aggregated temperature and UHII metrics
<div class="alert alert-block alert-success"> Using our `<b>calculate_uhii_enhanced</b>` function!</div>

### HIST

In [17]:
uhii_hist_urb_ext = calculate_uhii_extended(daily_stats_hist_urb,
                                            save_path=save_data_to,
                                            output_name="urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_hist_urb_ext.nc")

Processing 223 cities
Time period: 2017-01-01T00:00:00.000000000 to 2026-04-30T00:00:00.000000000


/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/cor

Excluding leap days (February 29)
Removed 2 leap days


In [18]:
uhii_hist_urb_ext

<xarray.Dataset> Size: 170MB
Dimensions:                   (time: 3405, city: 223)
Coordinates:
  * time                      (time) datetime64[ns] 27kB 2017-01-01 ... 2026-...
  * city                      (city) object 2kB 'Abu Dhabi' ... 'Zibo'
    location                  (city) <U2 2kB 'NH' 'NH' 'SH' ... 'NH' 'NH' 'NH'
    kg_class_main             (city) <U1 892B 'A' 'C' 'C' 'A' ... 'D' 'C' 'D'
    kg_class_detailed         (city) <U3 3kB 'As' 'Cwa' 'Csa' ... 'Csc' 'Dsd'
    kg_confidence_city        (city) float32 892B 100.0 92.27 ... 87.75 100.0
    lat                       (city) float64 2kB 24.24 8.993 ... 34.77 36.8
    lon                       (city) float64 2kB 54.68 38.77 ... 113.7 118.1
Data variables: (12/30)
    T2M_mean_mean_urban_high  (time, city) float64 6MB 289.7 287.6 ... 291.5
    T2M_mean_max_urban_high   (time, city) float64 6MB 290.1 287.6 ... 291.5
    T2M_mean_min_urban_high   (time, city) float64 6MB 289.3 287.6 ... 291.5
    T2M_mean_mean_urban_low   (time, city) float64 6MB 289.3 286.6 ... 291.1
    T2M_mean_max_urban_low    (time, city) float64 6MB 289.6 287.9 ... 292.1
    T2M_mean_min_urban_low    (time, city) float64 6MB 289.1 285.8 ... 289.3
    ...                        ...
    T2M_min_mean_rural        (time, city) float64 6MB 284.0 281.3 ... 283.7
    T2M_min_max_rural         (time, city) float64 6MB 286.0 284.3 ... 286.8
    T2M_min_min_rural         (time, city) float64 6MB 283.0 278.5 ... 281.7
    UHII                      (time, city) float64 6MB 0.7746 0.05591 ... 1.897
    n_urban_high_points       (city) int64 2kB 3 1 1 1 1 1 1 1 ... 1 0 2 1 1 1 1
    n_rural_points            (city) int64 2kB 20 40 20 40 20 ... 40 20 20 40 19

### CONT

In [19]:
uhii_cont_urb_ext = calculate_uhii_extended(daily_stats_cont_urb,
                                            save_path=save_data_to,
                                            output_name="urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_cont_urb_ext.nc")

Processing 224 cities
Time period: 2017-01-01T00:00:00.000000000 to 2026-04-30T00:00:00.000000000


/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/cor

Excluding leap days (February 29)
Removed 2 leap days


### T2K

In [20]:
uhii_t2k_urb_ext = calculate_uhii_extended(daily_stats_t2k_urb,
                                            save_path=save_data_to,
                                            output_name="urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_t2k_urb_ext.nc")

Processing 224 cities
Time period: 2017-01-01T00:00:00.000000000 to 2026-04-30T00:00:00.000000000


/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/core.py:917: RuntimeWarning: All-NaN slice encountered
  data = func(self.data, axis=axis, **kwargs)
/home/b/b383801/.conda/envs/ESDSRS/lib/python3.14/site-packages/xarray/namedarray/cor

Excluding leap days (February 29)
Removed 2 leap days


### Load saved data

In [21]:
uhii_hist_urb_ext = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_hist_urb_ext.nc")
uhii_cont_urb_ext = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_cont_urb_ext.nc")
uhii_t2k_urb_ext = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_t2k_urb_ext.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


## Remove cities without any urban_high points and/or any rural points from data set

In [22]:
print(uhii_hist_urb_ext.city.where(uhii_hist_urb_ext.n_urban_high_points == 0, drop=True).values)

['Ashaiman' 'Brookhaven' 'Changzhou' 'Coral Springs' 'Fort Worth' 'Mesa'
 'Pompano Beach' 'Riverside' 'Smithtown' 'Sosnowiec' 'Suzhou'
 'West Palm Beach' 'Wuxi']


In [23]:
print(uhii_hist_urb_ext.city.where(uhii_hist_urb_ext.n_rural_points == 0, drop=True).values)

[]


In [24]:
uhii_hist_urb_ext = uhii_hist_urb_ext.where(uhii_hist_urb_ext.n_urban_high_points > 0, drop=True)
uhii_cont_urb_ext = uhii_cont_urb_ext.where(uhii_cont_urb_ext.n_urban_high_points > 0, drop=True)
uhii_t2k_urb_ext = uhii_t2k_urb_ext.where(uhii_t2k_urb_ext.n_urban_high_points > 0, drop=True)
uhii_hist_urb_ext = uhii_hist_urb_ext.where(uhii_hist_urb_ext.n_rural_points > 0, drop=True)
uhii_cont_urb_ext = uhii_cont_urb_ext.where(uhii_cont_urb_ext.n_rural_points > 0, drop=True)
uhii_t2k_urb_ext = uhii_t2k_urb_ext.where(uhii_t2k_urb_ext.n_rural_points > 0, drop=True)

In [25]:
uhii_hist_urb_ext.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_hist_urb_ext.nc")
uhii_cont_urb_ext.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_cont_urb_ext.nc")
uhii_t2k_urb_ext.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_t2k_urb_ext.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


## Create heatwave mask

### Calculate percentile thresholds, extreme heat days mask
<div class="alert alert-block alert-success"> Using our `<b>identify_extreme_heat_pooling</b>` function!</div>

Pooling percentiles with moving window = <b>15 days</b> (120 values for each DOY) 

#### HIST

In [31]:
extreme_heat_hist_uhii, thresholds_heat_hist_uhii = identify_extreme_heat_pooling(uhii_hist_urb_ext,
                                                                                  reference_thresholds=None)

Calculating extreme heat days for ['T2M_mean_mean_urban_high', 'T2M_mean_mean_rural']
Excluding leap days (February 29)
Removed 0 leap days

Processing variable: T2M_mean_mean_urban_high
Calculating 90th percentile
Calculating 95th percentile
Calculating 98th percentile
Identifying days exceeding 90th percentile
Identifying days exceeding 95th percentile
Identifying days exceeding 98th percentile

Processing variable: T2M_mean_mean_rural
Calculating 90th percentile
Calculating 95th percentile
Calculating 98th percentile
Identifying days exceeding 90th percentile
Identifying days exceeding 95th percentile
Identifying days exceeding 98th percentile

Summary of extreme days:
T2M_mean_mean_urban_high, 90th percentile: 69,654 days (9.74%)
T2M_mean_mean_urban_high, 95th percentile: 34,442 days (4.82%)
T2M_mean_mean_urban_high, 98th percentile: 13,329 days (1.86%)
T2M_mean_mean_rural, 90th percentile: 69,664 days (9.74%)
T2M_mean_mean_rural, 95th percentile: 34,378 days (4.81%)
T2M_mean_mean_

#### CONT

In [32]:
extreme_heat_cont_uhii, _ = identify_extreme_heat_pooling(uhii_cont_urb_ext,
                                                          reference_thresholds=thresholds_heat_hist_uhii)

Calculating extreme heat days for ['T2M_mean_mean_urban_high', 'T2M_mean_mean_rural']
Excluding leap days (February 29)
Removed 0 leap days

Processing variable: T2M_mean_mean_urban_high
Identifying days exceeding 90th percentile
Identifying days exceeding 95th percentile
Identifying days exceeding 98th percentile

Processing variable: T2M_mean_mean_rural
Identifying days exceeding 90th percentile
Identifying days exceeding 95th percentile
Identifying days exceeding 98th percentile

Summary of extreme days:
T2M_mean_mean_urban_high, 90th percentile: 28,333 days (3.96%)
T2M_mean_mean_urban_high, 95th percentile: 11,562 days (1.62%)
T2M_mean_mean_urban_high, 98th percentile: 3,647 days (0.51%)
T2M_mean_mean_rural, 90th percentile: 27,807 days (3.89%)
T2M_mean_mean_rural, 95th percentile: 11,287 days (1.58%)
T2M_mean_mean_rural, 98th percentile: 3,507 days (0.49%)


#### T2K

In [33]:
extreme_heat_t2k_uhii, _ = identify_extreme_heat_pooling(uhii_t2k_urb_ext,
                                                        reference_thresholds=thresholds_heat_hist_uhii)

Calculating extreme heat days for ['T2M_mean_mean_urban_high', 'T2M_mean_mean_rural']
Excluding leap days (February 29)
Removed 0 leap days

Processing variable: T2M_mean_mean_urban_high
Identifying days exceeding 90th percentile
Identifying days exceeding 95th percentile
Identifying days exceeding 98th percentile

Processing variable: T2M_mean_mean_rural
Identifying days exceeding 90th percentile
Identifying days exceeding 95th percentile
Identifying days exceeding 98th percentile

Summary of extreme days:
T2M_mean_mean_urban_high, 90th percentile: 191,218 days (26.74%)
T2M_mean_mean_urban_high, 95th percentile: 125,323 days (17.53%)
T2M_mean_mean_urban_high, 98th percentile: 77,410 days (10.83%)
T2M_mean_mean_rural, 90th percentile: 195,847 days (27.39%)
T2M_mean_mean_rural, 95th percentile: 130,419 days (18.24%)
T2M_mean_mean_rural, 98th percentile: 81,615 days (11.41%)


## Create heatwave mask 
We define a heatwave as at least three consecutive extreme hot days.
<div class="alert alert-block alert-success"> Using our `<b>identify_heatwaves</b>` function!</div>

### HIST

In [34]:
heatwave_hist_uhii = identify_heatwaves(extreme_heat_hist_uhii)


Summary of heatwaves:
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p90): 69,654
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p90): 33,839
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p90): 4.73%
Percentage of heat days that belong to heatwaves (T2M_mean_mean_urban_high_extreme_p90): 48.58%
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p95): 34,442
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p95): 11,522
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p95): 1.61%
Percentage of heat days that belong to heatwaves (T2M_mean_mean_urban_high_extreme_p95): 33.45%
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p98): 13,329
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p98): 1,683
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p98): 0.24%
Percentage of heat days that belong to heatwaves (T2M_mea

### CONT

In [35]:
heatwave_cont_uhii = identify_heatwaves(extreme_heat_cont_uhii)


Summary of heatwaves:
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p90): 28,333
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p90): 10,039
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p90): 1.40%
Percentage of heat days that belong to heatwaves (T2M_mean_mean_urban_high_extreme_p90): 35.43%
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p95): 11,562
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p95): 2,412
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p95): 0.34%
Percentage of heat days that belong to heatwaves (T2M_mean_mean_urban_high_extreme_p95): 20.86%
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p98): 3,647
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p98): 179
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p98): 0.03%
Percentage of heat days that belong to heatwaves (T2M_mean_me

### T2K

In [36]:
heatwave_t2k_uhii = identify_heatwaves(extreme_heat_t2k_uhii)


Summary of heatwaves:
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p90): 191,218
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p90): 140,854
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p90): 19.70%
Percentage of heat days that belong to heatwaves (T2M_mean_mean_urban_high_extreme_p90): 73.66%
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p95): 125,323
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p95): 82,680
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p95): 11.56%
Percentage of heat days that belong to heatwaves (T2M_mean_mean_urban_high_extreme_p95): 65.97%
Extreme heat days total (T2M_mean_mean_urban_high_extreme_p98): 77,410
Heatwave days (>= 3 consecutive) (T2M_mean_mean_urban_high_extreme_p98): 44,928
Percentage of all days that are heatwave days (T2M_mean_mean_urban_high_extreme_p98): 6.28%
Percentage of heat days that belong to heatwaves (T

## Merge datasets
To create one final dataset with all variables (Mean/Max/Min Temperatures per celltype, UHII, Heat days thresholds, Heat day / wave mask)

### HIST

In [37]:
uhii_hist_urb_all_var = xr.merge([uhii_hist_urb_ext, extreme_heat_hist_uhii, heatwave_hist_uhii],
                                compat="no_conflicts", join="exact")
uhii_hist_urb_all_var.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_hist_urb_all_var_pooled15.nc", mode="w", engine="netcdf4")

### CONT

In [38]:
uhii_cont_urb_all_var = xr.merge([uhii_cont_urb_ext, extreme_heat_cont_uhii, heatwave_cont_uhii],
                                compat="no_conflicts", join="exact")
uhii_cont_urb_all_var.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_cont_urb_all_var_pooled15.nc", mode="w", engine="netcdf4")

### T2K

In [39]:
uhii_t2k_urb_all_var = xr.merge([uhii_t2k_urb_ext, extreme_heat_t2k_uhii, heatwave_t2k_uhii],
                                compat="no_conflicts", join="exact")
uhii_t2k_urb_all_var.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_t2k_urb_all_var_pooled15.nc", mode="w", engine="netcdf4")

### Load saved data

In [41]:
uhii_hist_urb_all_var = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_hist_urb_all_var_pooled15.nc")
uhii_cont_urb_all_var = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_cont_urb_all_var_pooled15.nc")
uhii_t2k_urb_all_var = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_t2k_urb_all_var_pooled15.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
